## 1. DEFINICIÓN Y LECTURA DE PARÁMETROS DEL PROYECTO

In [0]:
dbutils.widgets.removeAll()

# Diccionario centralizado con la configuración de todos los widgets
# Formato -> 'nombre_variable': ('valor_por_defecto', 'Etiqueta Visual')
project_config = {
    "cat_name": ("iowa_sales", "Catalog Name"),
    "sch_bronze": ("sales_bronze", "Bronze Schema Name"),
    "sch_silver": ("sales_silver", "Silver Schema Name"),
    "sch_gold": ("sales_gold", "Gold Schema Name"),
    "vol_process": ("/Volumes/iowa_sales/sales_bronze/process", "Source Process Path"),
    "vol_processed": ("/Volumes/iowa_sales/sales_bronze/processed", "Source Processed Path"),
    "tb_bronze": ("sales_raw", "Bronze Table Name"),
    "tb_silver": ("sales_cleaned", "Silver Table Name"),  
    "tb_fact": ("fact_sales", "Gold Fact Table Name"),
    "tb_dim_date": ("dim_date", "Date Dimension Table Name"),
    "tb_dim_vendor": ("dim_vendor", "Vendor Dimension Table Name"),
    "tb_dim_store": ("dim_store", "Store Dimension Table Name"),
    "tb_dim_prod": ("dim_product", "Product Dimension Table Name")
}

# Creación dinámica de los widgets en la UI de Databricks
for key, (default_val, label) in project_config.items():
    dbutils.widgets.text(key, default_val, label)

# Extracción de los valores en un diccionario de variables de entorno
env = {key: dbutils.widgets.get(key) for key in project_config.keys()}

In [0]:
# Lista de sentencias DDL generadas dinámicamente
ddl_statements = [
    f"CREATE CATALOG IF NOT EXISTS {env['cat_name']} COMMENT 'Iowa Liquor Sales Catalog'",
    f"CREATE SCHEMA IF NOT EXISTS {env['cat_name']}.{env['sch_bronze']} COMMENT 'Raw Data Layer'",
    f"CREATE SCHEMA IF NOT EXISTS {env['cat_name']}.{env['sch_silver']} COMMENT 'Cleansed Data Layer'",
    f"CREATE SCHEMA IF NOT EXISTS {env['cat_name']}.{env['sch_gold']} COMMENT 'Curated Star Schema Layer'",
    f"CREATE VOLUME IF NOT EXISTS {env['cat_name']}.{env['sch_bronze']}.process COMMENT 'Landing zone'",
    f"CREATE VOLUME IF NOT EXISTS {env['cat_name']}.{env['sch_bronze']}.processed COMMENT 'Archive zone'"
]

print("--- Inicializando infraestructura en Unity Catalog ---")
for statement in ddl_statements:
    # Imprimimos solo la parte principal del comando para mantener los logs limpios
    print(f"Ejecutando: {statement.split('COMMENT')[0].strip()}...")
    spark.sql(statement)

print("--- Configuración base completada exitosamente. ---")

## 2. CONFIGURACIÓN DE LA INFRAESTRUCTURA (CATALOG, SCHEMAS, VOLUMES)

In [0]:
import traceback
from pyspark.sql.functions import col, year, month, dayofmonth, date_format, quarter, dayofweek, weekofyear, lit

try:
    target_table = f"{env['cat_name']}.{env['sch_gold']}.{env['tb_dim_date']}"
    print(f"\nGenerando la dimensión de tiempo: {target_table} ...")
    
    # Usamos PySpark para generar la secuencia base de fechas
    base_dates_df = spark.sql("SELECT explode(sequence(TO_DATE('2012-01-01'), TO_DATE('2025-12-31'), INTERVAL 1 DAY)) AS full_date")
    
    # Construimos el DataFrame con todas las columnas requeridas usando la API de funciones
    dim_date_df = base_dates_df.select(
        (year("full_date") * 10000 + month("full_date") * 100 + dayofmonth("full_date")).alias("date_key"),
        col("full_date"),
        dayofmonth("full_date").alias("day"),
        month("full_date").alias("month"),
        date_format("full_date", "MMMM").alias("month_name"),
        quarter("full_date").alias("quarter"),
        year("full_date").alias("year"),
        dayofweek("full_date").alias("day_of_week"),
        date_format("full_date", "EEEE").alias("day_name"),
        weekofyear("full_date").alias("week_of_year"),
        lit(False).alias("is_a_holliday") # Mantenemos el nombre de tu columna original
    )

    # Escribimos el resultado directamente como una tabla Delta (modo Overwrite)
    dim_date_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(target_table)
    
    # Agregamos el comentario a la tabla
    spark.sql(f"ALTER TABLE {target_table} SET TBLPROPERTIES ('comment' = 'Date Dimension')")

    print(f"Dimensión {target_table} creada y poblada correctamente mediante PySpark API.")

except Exception as e:
    print(f"Error crítico al procesar la dimensión de tiempo: {str(e)}")
    print(traceback.format_exc())
    raise e